In [2741]:
import pandas as pd
from pathlib import Path
import numpy as np

import altair as alt

In [2742]:
root = "/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch/DataFrames/" # MAC
#root = "Y:/_Tobias/LSM980/Mitotic_Stopwatch/DataFrames/" # PC

df = pd.read_csv(root + "MainDataFrame_MitoticStopwatch_Augmin.csv")
df_eval = pd.read_csv(root + "MainDataFrame_MitoticStopwatch_Augmin_only_errors.csv") # check Multipolar Divition = object datatype???

dropped_cols = ['Unnamed: 0.1', 'Unnamed: 0', 'Track_ID', 'Source_ID',
       'Splitting_event', 'Lineage', 'Track_Coordinate_X', 'Generation', 'Mother_ID', 'Grandmother_ID', 'Sister_ID', 'Seen_sister', 'Seen_granny', 'Label', 'X',
       'Y', 'Comments',
       'Track_Coordinate_Y', ]


# df = df[df["Dataset"] == 20250724]
df = df.drop(dropped_cols, axis = 1)
df_eval = df_eval.drop(dropped_cols, axis = 1)
df.columns

Index(['Frame', 'Position', 'Dataset', 'Cell_Cycle_mins', 'Cell_Cycle_hours',
       ' ', 'Frame_onset', 'Frames_in_mitosis', 'Mitotic_duration_mins',
       'Interphase_duration_mins', 'Interphase_duration_hrs',
       'Mitotic_duration_hrs', 'Mother_Mitotic_duration_mins',
       'Mother_arrested', 'Metaphase_arrested', 'Frame_onset_hrs', 'Frame_hrs',
       'Augmin_depletion_bin', 'Mitotic_duration_bin', 'Unique_Track_ID',
       'Unique_Source_ID', 'Condition', 'Laggings', 'Massive Missegregation',
       'DNA bridge', 'Misaligned', 'MN', 'Multipolar Divition',
       'Citokinesis Failure', 'Slippage', 'Mitotic Death'],
      dtype='object')

In [2743]:
cols = [
    "Laggings", 
    "Massive Missegregation", 
    "DNA bridge", 
    "Misaligned", 
    "Multipolar Divition", 
    'Citokinesis Failure',
    'Slippage'
       ]

# Work on a copy to avoid warnings
bool_df = df_eval.loc[:, cols].copy()

# Convert and cast to nullable boolean
bool_df = bool_df.apply(lambda col: col.astype("boolean"))

print(bool_df.dtypes)

Laggings                  boolean
Massive Missegregation    boolean
DNA bridge                boolean
Misaligned                boolean
Multipolar Divition       boolean
Citokinesis Failure       boolean
Slippage                  boolean
dtype: object


In [2744]:
# Convert values: keep empty as NaN, only "True"/True as True, "False"/False as False
def to_bool_or_nan(x):
    if x in [True, "True"]:
        return True
    elif x in [False, "False"]:
        return False
    else:
        return np.nan   # preserve empties

bool_df = bool_df.map(to_bool_or_nan)

print(bool_df.dtypes)

Laggings                  bool
Massive Missegregation    bool
DNA bridge                bool
Misaligned                bool
Multipolar Divition       bool
Citokinesis Failure       bool
Slippage                  bool
dtype: object


In [2745]:
bool_df.loc[:, "any_true"] = bool_df.any(axis = 1, bool_only = True)

# New column: True if any True, NaN if all are NaN, otherwise False
df_eval.loc[:, "any_true"] = bool_df.any(axis = 1)

print(df_eval["any_true"].value_counts(dropna = False))

any_true
False    215
True      87
Name: count, dtype: int64


In [2746]:
df_augmin = df[df["Condition"] == "siHAUS6"]
df_ctrl = df[df["Condition"] == "siLUC1"]

In [2747]:
zero_df = df[df["Cell_Cycle_hours"] < 1]
zero_df

,Frame,Position,Dataset,Cell_Cycle_mins,Cell_Cycle_hours,,Frame_onset,Frames_in_mitosis,Mitotic_duration_mins,Interphase_duration_mins,...,Condition,Laggings,Massive Missegregation,DNA bridge,Misaligned,MN,Multipolar Divition,Citokinesis Failure,Slippage,Mitotic Death


In [2748]:
negative_df = df[df["Mother_Mitotic_duration_mins"] < 0]
negative_df

,Frame,Position,Dataset,Cell_Cycle_mins,Cell_Cycle_hours,,Frame_onset,Frames_in_mitosis,Mitotic_duration_mins,Interphase_duration_mins,...,Condition,Laggings,Massive Missegregation,DNA bridge,Misaligned,MN,Multipolar Divition,Citokinesis Failure,Slippage,Mitotic Death


In [2749]:
colourscheme = "viridis"

In [2750]:
def Binned_Boxplot(
    data, x, y, x_title, y_title, color, column, 
    Boxsize = 10,  
    Bin_Boxplot_width = 500, 
    Bin_Boxplot_height = 200):
    # Boxplot for binned x data
    BINNED_BOX = alt.Chart(
        data = data, 
        width = Bin_Boxplot_width, 
        height = Bin_Boxplot_height
    ).mark_boxplot(
        size = Boxsize, 
        extent = "min-max"
    ).encode(
    alt.X(x, title = None, axis = alt.Axis(grid = False)),
    alt.Y(y,  title = y_title, axis = alt.Axis(grid = False)),
    color = alt.Color(
        color, scale = alt.Scale(scheme = colourscheme)
    ),
    column = column
    ).configure_facet(
            spacing = 15
    ).configure_axis(
                grid = True, ticks = True, labelPadding = 5, offset = 10
    ).configure_header(
            labelOrient = 'bottom', title = None
    ).configure_view(
            stroke = 'transparent', 
            strokeWidth = 0.5
    )
    return BINNED_BOX

In [2751]:
def Binned_Boxplot2(
    data, x, y, x_title, y_title, color, column, 
    Boxsize = 10,  
    Bin_Boxplot_width = 40, 
    Bin_Boxplot_height = 200):
    # Boxplot for binned x data
    BINNED_BOX = alt.Chart(
        data = data, 
        width = Bin_Boxplot_width, 
        height = Bin_Boxplot_height
    ).mark_boxplot(
        size = Boxsize, 
        extent = "min-max"
    ).encode(
    alt.X(x, title = None, axis = alt.Axis(grid = False)),
    alt.Y(y,  title = y_title, axis = alt.Axis(grid = False)),
    color = alt.Color(
        color, scale = alt.Scale(scheme = colourscheme)
    ),
    column = column
    ).configure_facet(
            spacing = 15
    ).configure_axis(
                grid = True, ticks = True, labelPadding = 5, offset = 12
    ).configure_header(
            labelOrient = 'bottom', title = None
    ).configure_view(
            stroke = 'transparent', 
            strokeWidth = 0.5
    )
    return BINNED_BOX

In [2752]:
def stripbox_full(data, x, y, y_title, y_min, y_max, colour):
    boxplot = alt.Chart().mark_boxplot(
        extent = 'min-max', 
        size = 12
    ).encode(
        y = alt.Y(y, title = y_title, scale = alt.Scale(domain = [y_min, y_max])),
        opacity = alt.value(1),
        stroke = alt.value('black'),
        color = alt.value('white')
    ).properties(
        width = 18,
        height = 150
    )

    stripplot = alt.Chart().mark_circle(
        size = 20, opacity = 1
    ).encode(
        x = alt.X(
            'jitter:Q',
            title = None,
            axis = alt.Axis(values = [0], grid = False, labels = False, ticks = True),
        ),
        y = alt.Y(y, title = y_title, scale = alt.Scale(domain = [y_min, y_max]), 
            axis = alt.Axis(grid = False, labels = True, ticks = True)),
        color = alt.Color(colour, scale = alt.Scale(scheme = colourscheme))
        ).transform_calculate(
            jitter = '(sqrt(-2 * log(random() / 2)) * cos(2 * PI * random() / 2))'
    ).properties(
        width = 18,
        height = 150
    )
    
    FACETCHART = alt.layer(
        stripplot, boxplot, data = data
        ).facet(
            column = alt.Column(x, header = alt.Header(
                labelAngle = -90,
                titleOrient = 'top',
                labelOrient = 'bottom',
                labelAlign = 'right',
                labelPadding = 5)
                )
        ).configure_facet(
            spacing = 18
        ).configure_axis(
                grid = True, ticks = True, labelPadding = 5, offset = 15
        ).configure_header(
            labelOrient = 'bottom', title = None
        ).configure_view(
            stroke = 'transparent', 
            strokeWidth = 0.5
        )
    return FACETCHART

In [2753]:
def Binned_Circle(
    data, x, y, x_title, y_title, color, column, 
    Circlesize = 125,  
    Bin_Boxplot_width = 500, 
    Bin_Boxplot_height = 150,
    RawPointSize = 18  # Size for individual points
):

    # Shared encodings
    base_x = alt.X(x, title = None, axis = alt.Axis(grid = False))
    base_color = alt.Color(color, scale = alt.Scale(scheme = colourscheme))

    # Chart 1: Mean Circle
    mean = alt.Chart(data, width = Bin_Boxplot_width, height = Bin_Boxplot_height).mark_circle(size = Circlesize).encode(
        x = base_x,
        y = alt.Y(f"mean({y})", title = y_title, axis = alt.Axis(grid = False)),
        color = base_color
    )

    # Chart 2: Mean Error Bar
    mean_sem = alt.Chart(data, width=Bin_Boxplot_width, height=Bin_Boxplot_height).mark_errorbar().encode(
        x = base_x,
        y = alt.Y(f"mean({y})", title = y_title, axis = alt.Axis(grid = False)),
        color = base_color
    )

    # Chart 3: Raw Data Points
    raw_points = alt.Chart(data, width = Bin_Boxplot_width, height = Bin_Boxplot_height).mark_circle(size = RawPointSize, opacity = 0.1).encode(
        x = base_x,
        y = alt.Y(y, title = y_title, axis = alt.Axis(grid = False)),
        color = base_color
    )

    # Layer all three
    layered = raw_points + mean_sem + mean

    # Apply facet to combined chart
    faceted = layered.facet(
        column = column,
        spacing = 15
    )

    # Configure chart
    return faceted.configure_axis(
                grid = True, ticks = True, labelPadding = 5, offset = 12
           ).configure_header(
                labelOrient = 'bottom', title = None
           ).configure_view(
                stroke = 'transparent', 
                strokeWidth = 0.5
           )

In [2754]:
def Binned_Circle2(
    data, x, y, x_title, y_title, color, column, 
    Circlesize = 125,  
    Bin_Boxplot_width = 100, 
    Bin_Boxplot_height = 150,
    RawPointSize = 18  # Size for individual points
):

    # Shared encodings
    base_x = alt.X(x, title = None, axis = alt.Axis(grid = False))
    base_color = alt.Color(color, scale = alt.Scale(scheme = colourscheme))

    # Chart 1: Mean Circle
    mean = alt.Chart(data, width = Bin_Boxplot_width, height = Bin_Boxplot_height).mark_circle(size = Circlesize).encode(
        x = base_x,
        y = alt.Y(f"mean({y})", title = y_title, axis = alt.Axis(grid = False)),
        color = base_color
    )

    # Chart 2: Mean Error Bar
    mean_sem = alt.Chart(data, width = Bin_Boxplot_width, height = Bin_Boxplot_height).mark_errorbar().encode(
        x = base_x,
        y = alt.Y(f"mean({y})", title = y_title, axis = alt.Axis(grid = False)),
        color = base_color
    )

    # Chart 3: Raw Data Points
    raw_points = alt.Chart(data, width = Bin_Boxplot_width, height = Bin_Boxplot_height).mark_circle(size = RawPointSize, opacity = 0.1).encode(
        x = base_x,
        y = alt.Y(y, title = y_title, axis = alt.Axis(grid = False)),
        color = base_color
    )

    # Layer all three
    layered = raw_points + mean_sem + mean

    # Apply facet to combined chart
    faceted = layered.facet(
        column = column,
        spacing = 15
    )

    # Configure chart
    return faceted.configure_axis(
                grid = True, ticks = True, labelPadding = 5, offset = 12
           ).configure_header(
                labelOrient = 'bottom', title = None
           ).configure_view(
                stroke = 'transparent', 
                strokeWidth = 0.5
           )

In [2755]:
def Scatter_bin(dataframe, x, y, color, x_title, y_title, binextent, binstep,
            Circlesize = 15, 
            CircleOpacity = 0.2,  
            Scatter_width = 250, 
            Scatter_height = 200
               ):
    # Standard scatter plot 
    Scatter = alt.Chart(
        data = dataframe, 
        width = Scatter_width, 
        height = Scatter_height
    ).mark_circle(
        opacity = CircleOpacity,
        size = Circlesize
    ).encode(
        alt.X(x, title = x_title),
        alt.Y(y, title = y_title),
        color = alt.Color(
            color, legend = None, scale = alt.Scale(scheme = colourscheme)
        ) 
    )
    
    Scatter_binned = alt.Chart(
        data = dataframe, 
        width = Scatter_width, 
        height = Scatter_height
    ).mark_circle(
        opacity = 1,
        size = 100
    ).encode(
        alt.X(x, title = x_title, bin = alt.Bin(extent = binextent, step = binstep)),
        alt.Y(f'mean({y})', title = y_title, bin = False),
        color = alt.Color(
            color, scale = alt.Scale(scheme = colourscheme)
        ) 
    )
    
    Error_Scatterbinned = alt.Chart(
            data = dataframe
    ).mark_errorbar(extent = "stderr").encode(
        alt.X(x, title = x_title, bin = alt.Bin(extent = binextent, step = binstep)),
        alt.Y(f'mean({y})', title = y_title, bin = False),
        color = alt.Color(
            color, scale = alt.Scale(scheme = colourscheme)
        ) 
    ) 
    #SCATTERBIN = Scatter + Error_Scatterbinned + Scatter_binned 
    SCATTERBIN = Error_Scatterbinned + Scatter_binned
    return SCATTERBIN

In [2756]:
Stripbox_MitoticDurationbin = stripbox_full(
    data = df_augmin, 
    x = 'Mitotic_duration_bin', 
    y = 'Mitotic_duration_hrs', 
    y_title = 'Mitotic Duration (h)', 
    y_min = 0, 
    y_max = 40, 
    colour = "Condition"
)
Stripbox_MitoticDurationbin

alt.FacetChart(...)

In [2757]:
Stripbox_MitoticDuration = stripbox_full(
    data = df, 
    x = 'Condition', 
    y = 'Mitotic_duration_hrs', 
    y_title = 'Mitotic Duration (h)', 
    y_min = 0, 
    y_max = 40, 
    colour = "Condition"
)
Stripbox_MitoticDuration

alt.FacetChart(...)

In [2758]:
Stripbox_MitoticDuration = stripbox_full(
    data = df[df["Dataset"] == 20250807], 
    x = 'Condition', 
    y = 'Mitotic_duration_hrs', 
    y_title = 'Mitotic Duration (h)', 
    y_min = 0, 
    y_max = 40, 
    colour = "Condition"
)
Stripbox_MitoticDuration

alt.FacetChart(...)

In [2759]:
Stripbox_Interphase = stripbox_full(
    data = df, 
    x = 'Condition', 
    y = 'Interphase_duration_hrs', 
    y_title = 'Interphase Duration (h)', 
    y_min = 0, 
    y_max = 96, 
    colour = "Condition"
)
Stripbox_Interphase

alt.FacetChart(...)

In [2760]:
#Stripbox_Error = stripbox_full(
#    data = df[df["Segregation Error"].notnull()], 
#    y = 'Mitotic_duration_hrs', 
#    x = 'Segregation Error:N', 
#    y_title = 'Mitotic Duration (h)', 
#    y_min = 0, 
#    y_max = 30, 
#    colour = "Condition"
#)
#Stripbox_Error 

In [2761]:
# Step 1: Melt the DataFrame

df_rename = df.rename(columns = {"Interphase_duration_hrs": "2_Interphase_duration_hrs", "Mitotic_duration_hrs": "1_Mitotic_duration_hrs"})

df_long = pd.melt(
    df_rename[df_rename.Generation > 1],
    id_vars = ["Source_ID", "Condition"],
    value_vars = ["2_Interphase_duration_hrs", "1_Mitotic_duration_hrs"],
    var_name = "Phase",
    value_name = "Duration_hrs"
)

# Step 2: Ensure ID is numeric
df_long['Source_ID'] = df_long['Source_ID'].astype(int)

# Step 3: Compute total duration per ID
df_total = df_long.groupby(['Condition', 'Source_ID'], as_index = False)['Duration_hrs'].sum()
df_total.rename(columns = {'Duration_hrs': 'Total_duration'}, inplace = True)

# Step 4: Merge back into long DataFrame
df_long = df_long.merge(df_total, on = ['Condition', 'Source_ID'])

# Step 5: Create combined ID field
df_long['Condition_ID'] = df_long['Condition'] + "_" + df_long['Source_ID'].astype(str)

# Step 6: Sort Condition_IDs by total duration per condition
df_long['Condition_sort_key'] = df_long.groupby('Condition')['Total_duration'].rank(method = 'first', ascending = True)
df_long['SortKey'] = df_long['Condition'] + "_" + df_long['Condition_sort_key'].rank().astype(int).astype(str)

# Create a custom sort order
sorted_ids = df_long.sort_values(by = ['Condition', 'Total_duration'])['Condition_ID'].drop_duplicates().tolist()

# Step 7: Plot
bar_chart = alt.Chart(df_long).mark_bar().encode(
    x = alt.X('Condition_ID:N',
            sort = sorted_ids,
            axis = alt.Axis(labels = False, ticks = False, title = None)
           ),
    y = alt.Y('Duration_hrs:Q', stack = 'zero', title = 'Duration (hours)'),
    color = alt.Color('Phase:N', title = 'Cell Cycle Phase',
                    sort = alt.Sort(["2_Interphase_duration_hrs", "1_Mitotic_duration_hrs"]),  # Interphase at bottom
                    scale = alt.Scale(domain = ["2_Interphase_duration_hrs", "1_Mitotic_duration_hrs"],
                                    range = ["#a6cee3", "#1f78b4"])),
   # tooltip=['Condition', 'Source_ID', 'Phase', 'Duration_hrs']
).properties(
    width = 1000,
    height = 300
)
bar_chart

AttributeError: 'DataFrame' object has no attribute 'Generation'

In [ ]:
df.groupby(["Augmin_depletion_bin", "Condition"]).Metaphase_arrested.value_counts()

In [ ]:
df.groupby(["Augmin_depletion_bin", "Condition"]).Mitotic_duration_mins.mean()

In [ ]:
Scatter_MitoticDuration = Scatter_bin(
    dataframe = df,
    x = "Frame_onset_hrs",
    y = "Mitotic_duration_mins",
    color = "Condition",
    x_title = "Augmin depletion time (h)",
    y_title = "Mitotic duration (min)",
    binextent = [0, 96],
    binstep = 12
)
Scatter_MitoticDuration

In [ ]:
Scatter_MitoticDuration_hrs = Scatter_bin(
    dataframe = df,
    x = "Frame_onset_hrs",
    y = "Mitotic_duration_hrs",
    color = "Condition",
    x_title = "Augmin depletion time (h)",
    y_title = "Mitotic duration (hrs)",
    binextent = [0, 96],
    binstep = 12
)
Scatter_MitoticDuration_hrs

In [ ]:
Scatter_MitoticDuration_hrs_exp = Scatter_bin(
    dataframe = df[df["Condition"] == "siHAUS6"],
    x = "Frame_onset_hrs",
    y = "Mitotic_duration_hrs",
    color = "Dataset:O",
    x_title = "Augmin depletion time (h)",
    y_title = "Mitotic duration (hrs)",
    binextent = [0, 96],
    binstep = 12
)
Scatter_MitoticDuration_hrs_exp

In [ ]:
df.groupby(["Augmin_depletion_bin", "Condition", "Mother_arrested"]).Cell_Cycle_hours.mean()

In [ ]:
Scatter_CC = Scatter_bin(
    dataframe = df,
    x = "Frame_onset_hrs",
    y = "Cell_Cycle_hours",
    color = "Condition",
    x_title = "Augmin depletion time (h)",
    y_title = "Cell cycle duration (hours)",
    binextent = [0, 96],
    binstep = 24
)
Scatter_CC

In [ ]:
Scatter_CCinterphase = Scatter_bin(
    dataframe = df,
    x = "Frame_onset_hrs",
    y = "Interphase_duration_hrs",
    color = "Condition",
    x_title = "Augmin depletion time (h)",
    y_title = "Interphase duration (hours)",
    binextent = [0, 96],
    binstep = 24
)
Scatter_CCinterphase

In [ ]:
Scatter_CCint_augmin = Scatter_bin(
    dataframe = df_augmin,
    x = "Frame_onset_hrs",
    y = "Interphase_duration_hrs",
    color = "Mother_arrested",
    x_title = "Augmin depletion time (h)",
    y_title = "Interphase duration (hours)",
    binextent = [0, 96],
    binstep = 12
)
Scatter_CCint_augmin

In [ ]:
df.groupby(["Condition", "Mother_arrested"]).Unique_Track_ID.nunique()

In [ ]:
df.groupby(["Condition", "Mother_arrested"]).Interphase_duration_hrs.mean()

In [ ]:
df.groupby(["Condition"]).Interphase_duration_hrs.mean()

In [ ]:
df.groupby(["Condition", "Mother_arrested"]).Mitotic_duration_mins.mean()

In [ ]:
df.groupby(["Condition"]).Mitotic_duration_mins.mean()

In [ ]:
Bincircle_arrest = Binned_Circle2(
    data = df[df["Cell_Cycle_hours"] > 1],
    x = 'Mother_arrested',
    y = 'Cell_Cycle_hours',
    x_title = '',
    y_title = 'Cell_Cycle_hours',
    color = 'Mother_arrested:O',
    column = 'Condition:O'
)
Bincircle_arrest

In [ ]:
Bincircle_arrest2 = Binned_Circle2(
    data = df[df["Cell_Cycle_hours"] > 1],
    x = 'Mother_arrested',
    y = 'Interphase_duration_hrs',
    x_title = '',
    y_title = 'Interphase_duration_hrs',
    color = 'Mother_arrested:O',
    column = 'Condition:O'
)
Bincircle_arrest2

In [ ]:
Stripbox_CCcondition = stripbox_full(
    data = df, 
    x = 'Condition', 
    y = 'Cell_Cycle_hours', 
    y_title = 'Daughter cell cycle duration (h)', 
    y_min = 0, 
    y_max = 96, 
    colour = "Mother_arrested"
)
Stripbox_CCcondition

In [ ]:
Bincircle_arrestCondition = Binned_Circle2(
    data = df,
    x = 'Condition',
    y = 'Cell_Cycle_hours',
    x_title = '',
    y_title = 'Daughter cell cycle duration (h)',
    color = 'Mother_arrested:O',
    column = 'Mother_arrested'
)
Bincircle_arrestCondition

In [ ]:
Stripbox_CC = stripbox_full(
    data = df[df["Cell_Cycle_hours"] > 1], 
    x = 'Mother_arrested', 
    y = 'Cell_Cycle_hours', 
    y_title = 'Daughter cell cycle duration (h)', 
    y_min = 0, 
    y_max = 96, 
    colour = "Condition"
)
Stripbox_CC

In [ ]:
df.groupby("Mother_arrested").Cell_Cycle_hours.mean()

In [ ]:
df.Mother_arrested.value_counts()

In [ ]:
def plot_Individual_Daughters(df):
    bar = alt.Chart(df).mark_bar(width = 2).encode(
        x = alt.X(
            "Unique_Source_ID:N",
            title = "Individual daughter cells",
            sort = alt.EncodingSortField(
                field = "Mother_Mitotic_duration_mins",
                order = "ascending"
            ),
            axis = alt.Axis(
                labels = False, 
                ticks = False
            )
        ),
        y = alt.Y("Mother_Mitotic_duration_mins:Q", title = "Mother mitotic duration (min)"),
        color = alt.Color("Interphase_duration_hrs:Q", title = "Daughter interphase duration (hours)", scale = alt.Scale(scheme = "viridis")),
    )

    chart = bar.properties(
        width = 350,
        height = 200,
    )

    return chart

In [ ]:
daughters_augmin = plot_Individual_Daughters(df_augmin[df_augmin["Mother_Mitotic_duration_mins"] > 0])
daughters_augmin

In [ ]:
daughters_ctrl = plot_Individual_Daughters(df_ctrl[df_ctrl["Mother_Mitotic_duration_mins"] > 0])
daughters_ctrl

In [ ]:
df.groupby("Condition").Unique_Track_ID.nunique()

In [ ]:
# TO DO filter out tracks or branches that wander out of FOV

# For now only looking at siHAUS6 data

gen1_high_mitotic = df_augmin[
    (df_augmin["Generation"] == 1) & # First division in track (a bit redundant with following criteria)
    (df_augmin["Mitotic_duration_mins"] > 110) & # Metaphase arrest longer than 100 min
    (df_augmin["Frame_hrs"] < 36) & # division happens during first 36 h of augmin depletion
    (df_augmin["Frame_onset_hrs"] > 12) # division happens after first 12 h of augmin depletion
]

track_ids_of_interest1 = gen1_high_mitotic["Unique_Track_ID"].unique() # What are the Track IDs of families that have these criteria

result_df_hi = df_augmin[df_augmin["Unique_Track_ID"].isin(track_ids_of_interest1)] # create new df that only contains these cell families

gen1_low_mitotic = df_augmin[
    (df_augmin["Generation"] == 1) & # First division in track (a bit redundant with following criteria)
    (df_augmin["Mitotic_duration_mins"] < 110) & # Metaphase arrest longer than 100 min
    (df_augmin["Frame_hrs"] < 36) & # division happens during first 36 h of augmin depletion
    (df_augmin["Frame_onset_hrs"] > 12) # division happens after first 12 h of augmin depletion
]

track_ids_of_interest2 = gen1_low_mitotic["Unique_Track_ID"].unique() # What are the Track IDs of families that have these criteria

result_df_lo = df_augmin[df_augmin["Unique_Track_ID"].isin(track_ids_of_interest2)] # create new df that only contains these cell 

result_df = pd.concat([result_df_hi, result_df_lo])

print(f"Number of siHAUS6 Track_IDs (unique) with first division Mitotic_duration_mins > 110 dividing before 36 hours: {len(track_ids_of_interest1)}")
print(f"Number of siHAUS6 Track_IDs (unique) with first division Mitotic_duration_mins < 110 dividing before 36 hours: {len(track_ids_of_interest2)}")

In [ ]:
# Get the unique Track_IDs in result_df
all_track_ids = result_df["Unique_Track_ID"].unique()
total_track_ids = len(all_track_ids)

# Group by Track_ID and get the max Generation per track
max_gen_by_track = result_df.groupby("Unique_Track_ID")["Generation"].max()

In [ ]:
# a) Only Generation = 1
only_gen1 = (max_gen_by_track == 1).sum() # How many cell families have 1 generation only
print(f"Track_IDs with max generation 1: {only_gen1}")

# b) Only Generations ≤ 2
only_gen2 = (max_gen_by_track == 2).sum() # How many cell families have 2 generations
print(f"Track_IDs with max generation 2: {only_gen2}")

# c) Only Generations ≤ 3
only_gen3 = (max_gen_by_track == 3).sum() # How many cell families have 3 generations
print(f"Track_IDs with max generation 3: {only_gen3}")

# Compute percentages
percent_only_gen1 = (only_gen1 / total_track_ids) * 100
percent_only_gen2 = (only_gen2 / total_track_ids) * 100
percent_only_gen3 = (only_gen3 / total_track_ids) * 100

# Print results
print(f"Total Track_IDs in selection: {total_track_ids}")
print(f"Percentage with ONLY Generation = 1: {percent_only_gen1:.2f}%") # Among the selected tracks (> 100, ...)
print(f"Percentage with Generation ≤ 2: {percent_only_gen2:.2f}%") # Among the selected tracks (> 100, ...)
print(f"Percentage with Generation ≤ 3: {percent_only_gen3:.2f}%") # Among the selected tracks (> 100, ...)

In [ ]:
def compute_generation_percentages(df, depletion_time_min, depletion_time_max):
    # Helper to compute percentages by dataset
    def calculate_percentages(sub_df, population_label):
        # Get max generation per Track_ID
        #grouped = sub_df.groupby(["Unique_Track_ID", "Dataset", "Condition", "Position"])["Generation"].max().reset_index()
        grouped = sub_df.groupby(["Unique_Track_ID", "Dataset", "Condition"])["Generation"].max().reset_index()

        # Compute category flags
        grouped["Max_Gen1"] = grouped["Generation"] == 1
        grouped["Max_Gen2"] = grouped["Generation"] == 2
        grouped["Max_Gen3"] = grouped["Generation"] == 3
        grouped["Population"] = population_label

        # Aggregate counts and percentages by Dataset
        summary = grouped.groupby(["Dataset", "Population", "Condition"]).agg(
            Total_Tracks = ("Unique_Track_ID", "count"),
            Tracks_Only_Gen1 = ("Max_Gen1", "sum"),
            Tracks_Only_Gen2 = ("Max_Gen2", "sum"),
            Tracks_Only_Gen3 = ("Max_Gen3", "sum")
        ).reset_index()

        # Calculate percentages
        summary["Max_Gen1_percent"] = 100 * summary["Tracks_Only_Gen1"] / summary["Total_Tracks"]
        summary["Max_Gen2_percent"] = 100 * summary["Tracks_Only_Gen2"] / summary["Total_Tracks"]
        summary["Max_Gen3_percent"] = 100 * summary["Tracks_Only_Gen3"] / summary["Total_Tracks"]

        return summary[[
            "Dataset", "Population", "Total_Tracks", "Condition",
            "Max_Gen1_percent", "Max_Gen2_percent", "Max_Gen3_percent"
        ]]

    # Mother cell prolonged mitotic arrest
    gen1_high = df[
        (df["Generation"] == 1) & 
        (df["Mitotic_duration_mins"] > 110) & 
        (df["Frame_hrs"] < depletion_time_max) &
        (df["Frame_onset_hrs"] > depletion_time_min)
    ]
    high_track_ids = gen1_high["Unique_Track_ID"].unique()
    print(f"The number of filtered tracks that have HIGH mitotic times: {len(high_track_ids)}")
    df_high = df[df["Unique_Track_ID"].isin(high_track_ids)]
    print(df_high.groupby("Condition").Unique_Track_ID.nunique())
    summary_high = calculate_percentages(df_high, ">110 min") # calculating percentages using the helper function

    # Mother cell little or no mitotic arrest
    gen1_low = df[
        (df["Generation"] == 1) & 
        (df["Mitotic_duration_mins"] < 110) & 
        (df["Frame_hrs"] < depletion_time_max) &
        (df["Frame_onset_hrs"] > depletion_time_min)
    ]
    low_track_ids = gen1_low["Unique_Track_ID"].unique()
    print(f"The number of filtered tracks that have LOW mitotic times: {len(low_track_ids)}")
    df_low = df[df["Unique_Track_ID"].isin(low_track_ids)]
    print(df_low.groupby("Condition").Unique_Track_ID.nunique())
    summary_low = calculate_percentages(df_low, "<110 min") # calculating percentages using the helper function

    # Combine results
    final_summary = pd.concat([summary_high, summary_low], ignore_index = True)

    # Reshape to long format for plotting
    final_long = final_summary.melt(
        id_vars = ["Dataset", "Population", "Total_Tracks", "Condition"],
        value_vars = ["Max_Gen1_percent", "Max_Gen2_percent", "Max_Gen3_percent"],
        var_name = "Metric",
        value_name = "Percentage"
    )

    return df_high, df_low, final_long

results_df_high, results_df_low, percentages_df = compute_generation_percentages(df, depletion_time_min = 12, depletion_time_max = 36)

results_df_high_error, results_df_low_error, percentages_df_error = compute_generation_percentages(df_eval, depletion_time_min = 12, depletion_time_max = 36) # TODO check

percentages_df.to_csv(root + "PercentagesDataFrame_MitoticStopwatch_Augmin.csv")

In [ ]:
def plot_percentage_comparison(x = "Population:N", 
                               color = "Dataset:O",
                               color_bar = "Metric:N",
                               offset = "Metric:N", 
                               y = "Percentage", 
                               y_title = "% of cell families", 
                               data = percentages_df
                              ):

    bar = alt.Chart(data).mark_bar(width = 15).encode(
        x = alt.X(x, title = "", axis = alt.Axis(labelAngle = -0)),
        y = alt.Y(f"mean({y}):Q", title = y_title),
        color = alt.Color(color_bar, title = "", scale = alt.Scale(scheme = "viridis")),
        xOffset = offset
    )

    error = alt.Chart(data).mark_errorbar(extent = "stderr").encode(
        x = alt.X(x),
        y = alt.Y(f"mean({y}):Q", title = y_title),
        color = alt.Color(color_bar),
        xOffset = offset
    )

    datasets = alt.Chart(data).mark_circle(width = 15).encode(
        x = alt.X(x, title = "", axis = alt.Axis(labelAngle = 0)),
        y = alt.Y(f"mean({y})", title = y_title),
        color = alt.Color(color, title = "", scale = alt.Scale(scheme = "viridis")),
        xOffset = offset
    )

    chart = (bar + datasets + error).properties(
        width = 200,
        height = 200,
    )

    return chart

chart_augmin = plot_percentage_comparison(
    data = percentages_df[percentages_df["Condition"] == "siHAUS6"]
)

# ignore positions with only one family after bisecting? 

chart_ctrl = plot_percentage_comparison(
    data = percentages_df[percentages_df["Condition"] == "siLUC1"]
)

chart_augmin

In [ ]:
# TODO check if this can be done more elegantly by merging

result_df = pd.concat([results_df_high, results_df_low])
result_df_error = pd.concat([results_df_high_error, results_df_low_error])

result_df.to_csv(root + "Filtered_Augmin.csv")
result_df_error.to_csv(root + "Filtered_Augmin_errors.csv")


# Compute the maximum generation per Unique_Track_ID
max_gen_per_track = result_df.groupby("Unique_Track_ID")["Generation"].max()
max_gen_per_track_error = result_df_error.groupby("Unique_Track_ID")["Generation"].max()

# Create a mapping of Track_ID to Daughter_dividing? status
daughter_dividing_map = (max_gen_per_track > 1)
daughter_dividing_map_error = (max_gen_per_track_error > 1)

# Map the status back to the original df
result_df["Daughter_dividing?"] = result_df["Unique_Track_ID"].map(daughter_dividing_map)
result_df_error["Daughter_dividing?"] = result_df_error["Unique_Track_ID"].map(daughter_dividing_map_error)

result_df["Daughter_dividing?"] = result_df["Daughter_dividing?"].astype(str)  # ensure it's a string for coloring
result_df_error["Daughter_dividing?"] = result_df_error["Daughter_dividing?"].astype(str)  

result_df = result_df[result_df["Frame_hrs"] < 36] # Double check, why this is needed
result_df_error = result_df_error[result_df_error["Frame_hrs"] < 36]

df_sub_augmin = result_df[result_df["Condition"] == "siHAUS6"]
df_sub_augmin_error = result_df_error[result_df_error["Condition"] == "siHAUS6"]

df_sub_ctrl = result_df[result_df["Condition"] == "siLUC1"]
df_sub_ctrl_error = result_df_error[result_df_error["Condition"] == "siLUC1"]

In [ ]:
df_sub_augmin.columns

In [ ]:
def plot_Individual_Daughters_arrested(df, barwidth):
    bar = alt.Chart(df).mark_bar(width = barwidth).encode(
        x = alt.X(
            "Unique_Source_ID:N",
            title = "Individual daughter cells",
            sort = alt.EncodingSortField(
                field = "Mitotic_duration_mins",
                order = "ascending"
            ),
            axis = alt.Axis(
                labels = False, 
                ticks = False
            )
        ),
    #    y = alt.Y("Mitotic_duration_mins:Q", title = "Mother mitotic duration (min)", scale = alt.Scale(domain = [0, 450]), bin = alt.Bin(maxbins = 50)),
        y = alt.Y("Mitotic_duration_mins:Q", title = "Mother mitotic duration (min)", scale = alt.Scale(domain = [0, 450])), 
        color = alt.Color("Daughter_dividing?", title = "Daughter dividing?", scale = alt.Scale(scheme = "viridis")),
    )

    chart = bar.properties(
        width = 375,
        height = 200,
    )

    return chart

In [ ]:
daughters_augmin = plot_Individual_Daughters_arrested(df_sub_augmin, barwidth = 2.5)

In [ ]:
daughters_augmin # TODO doublecheck the division with +400 min in mitosis

In [ ]:

df_sub_augmin["Daughter_dividing?"].value_counts()

In [ ]:
#TODO finish

def find_threshold(df = df_sub_augmin, duration_col = "Mitotic_duration_mins", outcome_col = "Daughter_dividing?", target = 0.05):
   
    df = df.copy()
    df[outcome_col] = df[outcome_col].astype(str).str.strip().map({"True": True, "False": False})

    
    # Sort by duration
    df_sorted = df.sort_values(by = duration_col)
    
    thresholds = df_sorted[duration_col].unique()
    
    for thresh in thresholds:
        subset = df_sorted[df_sorted[duration_col] >= thresh]
        if len(subset) == 0:
            continue
        frac_true = subset[outcome_col].mean()  # fraction of True values
        if frac_true >= target:
            return thresh, frac_true
    
    return None, None  # if no threshold meets the criterion

threshold, frac = find_threshold()
print(f"Threshold = {threshold}, Above threshold fraction True = {frac:.2f}")

In [ ]:
daughters_ctrl = plot_Individual_Daughters_arrested(df_sub_ctrl, barwidth = 3)
daughters_ctrl

In [ ]:
df_sub_ctrl["Daughter_dividing?"].value_counts()

In [ ]:
def plot_Individual_Daughters_arrested_bins(df, barwidth):
    bar = alt.Chart(df).mark_bar(width = barwidth).encode(
        x = alt.X(
            "Unique_Source_ID:N",
            title = "Individual daughter cells",
            sort = alt.EncodingSortField(
                field = "Mitotic_duration_bin",
                order = "ascending"
            ),
            axis = alt.Axis(
                labels = False, 
                ticks = False
            )
        ),
    #    y = alt.Y("Mitotic_duration_mins:Q", title = "Mother mitotic duration (min)", scale = alt.Scale(domain = [0, 450]), bin = alt.Bin(maxbins = 50)),
        y = alt.Y("Mitotic_duration_bin:Q", title = "Mother mitotic duration (binned)", scale = alt.Scale(domain = [0, 450])), 
        color = alt.Color("Daughter_dividing?", title = "Daughter dividing?", scale = alt.Scale(scheme = "viridis")),
    )

    chart = bar.properties(
        width = 375,
        height = 200,
    )

    return chart

In [ ]:
daughters_augmin = plot_Individual_Daughters_arrested_bins(df_sub_augmin, barwidth = 2.5)
daughters_augmin

In [ ]:
percentages_df[percentages_df["Condition"] == "siHAUS6"].groupby(["Dataset", "Condition", "Population"]).head()

In [ ]:
def plot_Individual_Daughters_error(df, category, barwidth, y_max, plotwidth):
    bar = alt.Chart(df).mark_bar(width = barwidth).encode(
        x = alt.X(
            "Unique_Source_ID:N",
            title = "Individual divisions",
            sort = alt.EncodingSortField(
                field = "Mitotic_duration_mins",
                order = "ascending"
            ),
            axis = alt.Axis(
                labels = False, 
                ticks = False
            )
        ),
        y = alt.Y("Mitotic_duration_mins:Q", title = "Mitotic duration (min)", scale = alt.Scale(domain = [0, y_max])), 
        color = alt.Color(category, title = f"{category}?", scale = alt.Scale(scheme = "viridis")),
    )

    chart = bar.properties(
        width = plotwidth,
        height = 200,
    )

    return chart

In [ ]:
df_eval_augmin = df_eval[df_eval["Condition"] == 'siHAUS6']

daughters_augmin_any = plot_Individual_Daughters_error(
    df_eval_augmin,
    category = "any_true",
    barwidth = 2.5,
    y_max = 1500,
    plotwidth = 480
)
daughters_augmin_any

In [ ]:
daughters_augmin_selected_any = plot_Individual_Daughters_error(
    df_sub_augmin_error,
    category = "any_true",
    barwidth = 5,
    y_max = 450,
    plotwidth = 300
)
daughters_augmin_selected_any

In [ ]:
daughters_augmin_death = plot_Individual_Daughters_error(
    df_eval_augmin,
    category = "Mitotic Death",
    barwidth = 2.5,
    y_max = 1500,
    plotwidth = 480
)
daughters_augmin_death

In [ ]:
daughters_augmin_selected_any = plot_Individual_Daughters_error(
    df_sub_augmin_error,
    category = "Mitotic Death",
    barwidth = 5,
    y_max = 450,
    y_max = 450,
    plotwidth = 300
)
daughters_augmin_selected_any

In [ ]:
df_eval_ctrl = df_eval[df_eval["Condition"] == 'siLUC1']

daughters_ctrl_any = plot_Individual_Daughters_error(
    df_eval_ctrl,
    category = "any_true",
    barwidth = 5,
    y_max = 1500,
    plotwidth = 480
)
daughters_ctrl_any

In [ ]:
daughters_ctrl_selected_any = plot_Individual_Daughters_error(
    df_sub_ctrl_error,
    category = "any_true",
    barwidth = 5,
    y_max = 450,
    plotwidth = 300
)
daughters_ctrl_selected_any

In [ ]:
df_sub_ctrl_error. # map True to daughter dividing

In [ ]:
# whats the percentage of cells where there daughter is arrested without any_true?

df_sub_augmin_error_noarrested_error = df_sub_augmin_error.loc[(df_sub_augmin_error["any_true"] == True) & (df_sub_augmin_error["Daughter_dividing?"] == "True"), :]
df_sub_augmin_error_noarrested_error

In [ ]:
df_sub_augmin_error.shape

In [ ]:
df_sub_augmin_error_arrested_noerror = df_sub_augmin_error.loc[(df_sub_augmin_error["any_true"] == False) & (df_sub_augmin_error["Daughter_dividing?"] == "False"), :]
df_sub_augmin_error_arrested_noerror.shape

In [ ]:
df_sub_augmin_error_arrested_noerror.Mitotic_duration_mins.mean()

In [ ]:
df_sub_augmin_error_noarrested_noerror = df_sub_augmin_error.loc[(df_sub_augmin_error["any_true"] == False) & (df_sub_augmin_error["Daughter_dividing?"] == "True"), :]
df_sub_augmin_error_noarrested_noerror.shape

In [ ]:
df_sub_augmin_error.columns

In [ ]:
df_short = df_eval[['Condition', 'Laggings', 'Massive Missegregation', 'DNA bridge', 'Misaligned', 'MN', 'Multipolar Divition', 'Citokinesis Failure', 'Slippage', 'Mitotic Death']]
df_short.head()

In [ ]:
list_of_categories = ['Laggings', 'Massive Missegregation', 'DNA bridge', 'Misaligned', 'Multipolar Divition', 'Citokinesis Failure', 'Slippage', 'Mitotic Death']

def get_error_percentages(df = df_short, categories = list_of_categories):
    list_of_conditions = df.Condition.unique()
    
    for condition in list_of_conditions:
        
        df_cond = df[df.Condition == condition]

        for category in categories:
            print(df[category]).value_counts()
            df[category] = df[category].replace({"True": True, "False": False}).astype("boolean")
            true_in_category = df_cond[category].sum()
            print(category)
            print(true_in_category)
        

get_error_percentages()